## FovealGAT: components overview

In [1]:
import only_gaze_confs as conf

choosing least busy GPU...
running on cuda:1


With the gaze patch features and encodings aggregated into graph nodes, we can feed it through our network. We'll go through each component.

Fetch sample batch

In [2]:
import torch
from gaze_datasets.reflacx import ReflacxGazePatchDataset
from gaze_datasets.dataset_tools import gaze_patch_dataset_collate
from feature_extraction.extractors import GazePatchFeatureExtractor
from node_encodings.sequence import GazeSinCosPosEncoding

gaze_dataset_pth = conf.reflacx_gaze_dataset_pth
desired_label = 'Atelectasis'
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label)
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

max_len = conf.reflacx_max_len
seq_encoding_featcount = conf.seq_encoding_feat_count

seq_encoding = GazeSinCosPosEncoding(max_len, seq_encoding_featcount)
seqs = seq_encoding(splits, adjust_dim=True).to(conf.device)
fixations = fixations.to(conf.device)

nodes = torch.cat([fixations, seqs, feats], 1).float()
nodes.shape, sum([x.shape[1]for x in (seqs, fixations, feats)])

(torch.Size([255, 147, 1, 1]), 147)

In [3]:
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))

### GAT Layer
<a id="gat_layer"></a>

The first component is the Graph Attention Convolution Layer.

In [4]:
from model.gat_conv import CausalGATConv

set number of attention heads for the layer

In [5]:
att_heads = 4

In [6]:
gcnn = CausalGATConv(ns.shape[0],
                     ns.shape[0],
                     att_heads,
                     device=conf.device)

In [7]:
conv_nodes = gcnn((ns, splits))
conv_nodes.shape

torch.Size([147, 255])

### XYTGATLayer

a gat layer that has attention mechanisms that look only into pair of nodex xy coord and pairs of time duration

In [8]:
from xyt_gat_conv import CausalXYTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices

In [9]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)

In [10]:
import torch

In [11]:
gcnn = CausalXYTGATConv(ns.shape[0],
                        ns.shape[0],
                        8,
                        device=conf.device)

In [12]:
gcnn

CausalXYTGATConv(
  (att_scores): Conv2d(294, 8, kernel_size=(1, 1), stride=(1, 1))
  (xy_scores): Conv2d(4, 2, kernel_size=(1, 1), stride=(1, 1))
  (t_scores): Conv2d(2, 1, kernel_size=(1, 1), stride=(1, 1))
  (node_update): Conv1d(1617, 147, kernel_size=(1,), stride=(1,))
  (norm): GraphNorm(147)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [13]:
conv_nodes = gcnn((ns, splits, xy_mats, t_mats))
conv_nodes.shape

torch.Size([147, 255])

### EuclideanTGATLayer

a gat layer that ads delta x, delta y and euclidean distance between two fixations to spatial attention adjacency matrix

In [14]:
from xyt_gat_conv import CausalEuclideanTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices, get_euclidean_encodings

In [15]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
xy_eucs = [get_euclidean_encodings(xym) for xym in xy_mats]
[e.shape for e in xy_eucs]

[torch.Size([7, 47, 47]), torch.Size([7, 155, 155]), torch.Size([7, 53, 53])]

In [16]:
import torch

In [17]:
gcnn = CausalEuclideanTGATConv(ns.shape[0],
                               ns.shape[0],
                               8,
                               device=conf.device)

In [18]:
gcnn

CausalEuclideanTGATConv(
  (att_scores): Conv2d(294, 8, kernel_size=(1, 1), stride=(1, 1))
  (xy_scores): Conv2d(7, 2, kernel_size=(1, 1), stride=(1, 1))
  (t_scores): Conv2d(2, 1, kernel_size=(1, 1), stride=(1, 1))
  (node_update): Conv1d(1617, 147, kernel_size=(1,), stride=(1,))
  (norm): GraphNorm(147)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [19]:
conv_nodes = gcnn((ns, splits, xy_eucs, t_mats))
conv_nodes.shape

torch.Size([147, 255])

### EuclideanSummaryTGATLayer

In [22]:
from xyt_gat_conv import XYTGATConv
import torch
from torch import nn
from torch.nn import functional as F
from torch_geometric.nn.norm import GraphNorm

from math import floor

from model.tools import init_weights

In [25]:
splits, ns.shape

([47, 155, 53], torch.Size([147, 255]))

In [ ]:
class CausalSummaryXYTGATConv(XYTGATConv):
    def __init__(self,
                 in_channels,
                 out_channels,
                 att_heads,
                 xy_heads_ratio=0.2,
                 t_heads_ratio=0.1,
                 act_att=F.leaky_relu,
                 act_gat=F.elu,
                 norm=GraphNorm,
                 dropout=0.1,
                 bias=True,
                 w_init_f=lambda w: nn.init.xavier_uniform_(w),
                 device='cpu'):
        super().__init__(in_channels,
                         out_channels,
                         att_heads,
                         xy_heads_ratio,
                         t_heads_ratio,
                         act_att,
                         act_gat,
                         norm,
                         dropout,
                         bias,
                         w_init_f,
                         device)
    """A GATConv layer for a causal graph,
    where each node sees only itself and their predecessors
    """
    def forward(self, x):
        nodes, splits, xy_mats, t_mats = x
        # normalize graph
        batch_idx = torch.cat([torch.ones(s, dtype=torch.int) * i
                               for i, s in enumerate(splits)]).to(self.device)
        nodes = torch.transpose(nodes, 1, 0)
        nodes = self.norm(nodes, batch_idx.to(torch.long))
        nodes = torch.transpose(nodes, 1, 0)
        
        #check if graphs have summary nodes and generate adj matrices
        if nodes.shape[-1] == sum(splits) + len(splits): #if previous graph has a summary node
            subnodes = torch.split(nodes, [s + 1 for s in splits], dim=1)
        
        # generate adjacency matrix for whole batch
        subnodes = torch.split(nodes, splits, dim=1)
        subnodes = [self.dropout(sn) for sn in subnodes]
        adjs = [XYTGATConv.adj_mat(ns) for ns in subnodes]

        # generate raw semantic scores
        sem_raw_scores = [self.act_att(self.att_scores(adj)) for adj in adjs]
        
        # generate raw XY scores
        xy_raw_scores = [self.act_att(self.xy_scores(xy_mat)) for xy_mat in xy_mats]

        # generate raw T scores
        t_raw_scores = [self.act_att(self.t_scores(t_mat)) for t_mat in t_mats]
        
        rawscores = [torch.cat((sem, xy, t), dim=0)
                     for sem, xy, t in zip(sem_raw_scores,
                                           xy_raw_scores,
                                           t_raw_scores)]        

        # set upper triangle to zero, so a node doesn't see future nodes
        adj_mask = [torch.tril(torch.ones(rs.shape,
                                dtype=torch.bool,
                                device=self.device))
                    for rs in rawscores]
        masked_scores = [torch.where(am,
                                     rs,
                                     torch.tensor(float('-inf'),
                                     device=self.device))
                         for am, rs in zip(adj_mask, rawscores)]

        # generate final att weights
        att_ws = [F.softmax(ms, dim=-1) for ms in masked_scores]

        # pass messages and generate output nodes
        gatconv = [torch.matmul(sn, s) for sn, s in zip(subnodes, att_ws)]
        gatconv = torch.cat(gatconv, -1)
        gatconv = gatconv.reshape((gatconv.shape[0] * gatconv.shape[1], gatconv.shape[2]))
        return self.act_gat(self.node_update(gatconv))

### GazeGrapher
<a id="grapher"></a>

GazeGrapher is a module that aggregates several GAT layers in sequence, performing all graph convolutions before graph readout and prediction.

PyTorch's native nn.Sequential does not keep residual tensors between its internal steps, so we need to extend it. Graph readout and predictions will need residual, intermediary, graphs

In [ ]:
from residual_sequential import MarkovResidualSequential, DenseResidualSequential

We need input sizes for each of the graphs GAT Layers, and the residual layers' indices and GAT conv class of choice

In [ ]:
layers = 4
gat_channels = 256 # input size and 4 GAT convolution layers
residual_gat_levels = [False, True, True, True]
att_heads = [16, 8, 8, 8]
output_gat_levels = None
gat_conv = CausalGATConv

In [ ]:
from grapher import GazeGrapher, XYTGazeGrapher

In [ ]:
from torch import nn
from torch_geometric.nn.norm import GraphNorm
import torch.nn.functional as F

from model.gat_conv import CausalGATConv

In [ ]:
grapher = GazeGrapher(layers=layers,
                      output_gat_levels=output_gat_levels,
                      residual_gat_levels=residual_gat_levels,
                      att_heads=att_heads,
                      gat_conv_args={'channels': gat_channels,
                                     'act_att': F.leaky_relu,
                                     'act_gat': F.elu,
                                     'norm': GraphNorm,
                                     'dropout': 0.1,
                                     'bias': True,
                                     'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                      device=conf.device)
grapher

In [ ]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

In [ ]:
graphs = grapher((ens, splits)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
grapher = XYTGazeGrapher(layers=layers,
                         output_gat_levels=output_gat_levels,
                         residual_gat_levels=residual_gat_levels,
                         att_heads=att_heads,
                         gat_conv_args={'channels': gat_channels,
                                        'xy_heads_ratio': 0.2,
                                        't_heads_ratio': 0.1,
                                        'act_att': F.leaky_relu,
                                        'act_gat': F.elu,
                                        'norm': GraphNorm,
                                        'dropout': 0.1,
                                        'bias': True,
                                        'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                         device=conf.device)
grapher

In [ ]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

In [ ]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
from grapher import EuclideanTGazeGrapher

In [ ]:
grapher = EuclideanTGazeGrapher(layers=layers,
                                output_gat_levels=output_gat_levels,
                                residual_gat_levels=residual_gat_levels,
                                att_heads=att_heads,
                                gat_conv_args={'channels': gat_channels,
                                               'xy_heads_ratio': 0.2,
                                               't_heads_ratio': 0.1,
                                               'act_att': F.leaky_relu,
                                               'act_gat': F.elu,
                                               'norm': GraphNorm,
                                               'dropout': 0.1,
                                               'bias': True,
                                               'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                                device=conf.device)
grapher

In [ ]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

In [ ]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

### Readout
<a id="readout"></a>

Since we are dealing with complete causal graphs --each node sees ALL of its predecessors and itself-- we developed a causal graph readout. All information is aggregated by the last node at each convolution. So, we take the last node of each graph selected by residual_gat_levels, concatenate them, and run them through linear layers, outputting the final predictions.

In [ ]:
from readout import CausalReadout

In [ ]:
sum([g.shape[0] for g in graphs])

In [ ]:
ro_channels=conf.ro_channels
ro_channels[0] = sum([g.shape[0] for g in graphs])

In [ ]:
causal_ro = CausalReadout(channels=ro_channels,
                          dropout=conf.ro_dropout,
                          device=conf.device)

In [ ]:
causal_ro

In [ ]:
predictions = causal_ro((graphs, splits))
predictions.shape, labels.shape

### Whole pipeline
<a id="full"></a>

Since we now have all individual components, we are ready to run the full model.

In [ ]:
from model.foveal_gat import FovealGAT

In [ ]:
gat_conv_args={'channels': gat_channels,
               'att_heads': 8,
               'act_att': F.leaky_relu,
               'act_gat': F.elu,
               'norm': GraphNorm,
               'dropout': 0.1,
               'bias': True,
               'w_init_f': lambda w: nn.init.xavier_uniform_(w)}

In [ ]:
model = FovealGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalGATConv,
                    gat_conv_args=gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [ ]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape

In [ ]:
xyt_gat_conv_args={'channels': gat_channels,
                   'att_heads': 8,
                   'xy_heads_ratio': 0.2,
                   't_heads_ratio': 0.1,
                   'act_att': F.leaky_relu,
                   'act_gat': F.elu,
                   'norm': GraphNorm,
                   'dropout': 0.1,
                   'bias': True,
                   'w_init_f': lambda w: nn.init.xavier_uniform_(w)}

In [ ]:
model = FovealGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalXYTGATConv,
                    gat_conv_args=xyt_gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [ ]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape

In [ ]:
euc_gat_conv_args={'channels': gat_channels,
                   'att_heads': 8,
                   'xy_heads_ratio': 0.2,
                   't_heads_ratio': 0.1,
                   'act_att': F.leaky_relu,
                   'act_gat': F.elu,
                   'norm': GraphNorm,
                   'dropout': 0.1,
                   'bias': True,
                   'w_init_f': lambda w: nn.init.xavier_uniform_(w)}

In [ ]:
model = FovealGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalEuclideanTGATConv,
                    gat_conv_args=euc_gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [ ]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape